# NeuroGolf 2026: 6151.18 Current-Rules Open Artifact

This notebook publishes my latest public submission artifact and the audit trail behind the May 31 improvement chain.

The important lesson changed from the original 5689.51 release: direct public grafting is still risky, but carefully audited public-output ONNX files can be valuable when they pass the refreshed current-rules scorer and do not hash-match known poisoned artifact families.

**Public score:** `6151.18`  
**Final artifact:** `submission.zip` with `400` ONNX files  
**Artifact SHA256:** `21B1D31241AF38A6B118A4C526A81A55509517F0CA6686F6C0F9A0A477842DB5`  
**Main theme:** current-rules validation, public-output attribution, single-task grafting, and conservative rejection of known bad artifact hashes.


## TL;DR

1. This notebook writes the exact `6151.18` artifact to `/kaggle/working/submission.zip`.
2. The artifact is self-contained in the notebook, then verified with a manifest: `400` ONNX files, byte lengths, and SHA256 hashes.
3. The May 31 chain moved the public score:

`6141.04 -> 6149.27 -> 6151.01 -> 6151.14 -> 6151.18`

4. The main wins were:

- `+8.23` local points from Chet's GLM/Opus notebook outputs on tasks `035,041,093,134,225,290,316,354,355,388`.
- `+1.74` local points from Biohack44's stronger `task258`.
- `+0.13` local points from fresh-hash Biohack44 v2 tasks `076,360`.
- `+0.039` local points from a post-graft cleanup pass.

5. A tempting Rauff/old-artifact family was rejected even though it looked locally strong, because the hashes matched prior online-negative artifacts.


In [ ]:
from pathlib import Path
import base64
import hashlib
import zipfile

import pandas as pd

DATASET = Path('/kaggle/input/neurogolf-5689-51-current-rules-open-artifact')
WORK = Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)

exact_zip_b64 = DATASET / 'submission_zip_b64.txt'
exact_zip_bytes = DATASET / 'submission_zip.bin'
manifest_path = DATASET / 'onnx_manifest.csv'
ledger_path = DATASET / 'submission.csv'

EXPECTED_SHA256 = '21B1D31241AF38A6B118A4C526A81A55509517F0CA6686F6C0F9A0A477842DB5'
EXPECTED_TASKS = 400

if not DATASET.exists():
    raise FileNotFoundError(
        'The artifact dataset is missing. Add dataset '
        '`afr1ste/neurogolf-5689-51-current-rules-open-artifact` to this notebook.'
    )

out_path = WORK / 'submission.zip'
if exact_zip_b64.exists():
    out_path.write_bytes(base64.b64decode(exact_zip_b64.read_text().strip()))
elif exact_zip_bytes.exists():
    out_path.write_bytes(exact_zip_bytes.read_bytes())
else:
    raise FileNotFoundError('Expected submission_zip_b64.txt or submission_zip.bin in the artifact dataset.')

actual_sha = hashlib.sha256(out_path.read_bytes()).hexdigest().upper()
assert actual_sha == EXPECTED_SHA256, (actual_sha, EXPECTED_SHA256)

manifest = pd.read_csv(manifest_path)
assert len(manifest) == EXPECTED_TASKS, f'Expected {EXPECTED_TASKS} ONNX files, found {len(manifest)}'

with zipfile.ZipFile(out_path) as zf:
    task_names = sorted(
        info.filename for info in zf.infolist()
        if not info.is_dir() and Path(info.filename).name.startswith('task') and Path(info.filename).suffix == '.onnx'
    )
assert len(task_names) == EXPECTED_TASKS, len(task_names)

print('Wrote', out_path)
print('Submission SHA256:', actual_sha)
print('Task count:', len(task_names))
manifest.head()


In [ ]:
chain = pd.DataFrame([
    {'version': 'v69', 'public_score': 6141.04, 'source': 'local structural rewrite', 'tasks': '044', 'delta_local': 0.065981850020, 'note': 'unit-leading Reshape -> Gather(axis=0,index=0) elision'},
    {'version': 'v70', 'public_score': 6149.27, 'source': 'jsrdcht/glm-vs-opus-onnx-cost-opt-neurogolf-2026', 'tasks': '035,041,093,134,225,290,316,354,355,388', 'delta_local': 8.231400610687, 'note': 'self-contained GLM-5.1/Opus optimized ONNX, all exact under current scorer'},
    {'version': 'v71', 'public_score': 6151.01, 'source': 'biohack44/neurogolf-2026-fp16-surgery-prune-blend-6115', 'tasks': '258', 'delta_local': 1.738270784277, 'note': 'selected Biohack cost-160 task258 over weaker Massimiliano cost-310 variant'},
    {'version': 'v72', 'public_score': 6151.14, 'source': 'biohack44/neurogolf-new-best-public-v2-6080-ish', 'tasks': '076,360', 'delta_local': 0.132096530175, 'note': 'fresh-hash positives with no old poisoned-artifact match'},
    {'version': 'v73', 'public_score': 6151.18, 'source': 'local cleanup after public grafts', 'tasks': '035,076,096,134,158,225,316,354', 'delta_local': 0.038881214906, 'note': 'initializer dedup plus Cast(bool)*Mul -> Where cleanup'},
])
chain


## What changed since the 5689.51 notebook

The original version of this notebook focused on rebuilding symbolic task compilers from public clues. That is still useful, but the stronger May 31 path was a stricter public-output audit:

- pull a public notebook or output attachment;
- evaluate every changed ONNX against the current local baseline;
- keep only `status=ok` tasks with positive current-rules points;
- check hashes against known bad public artifact families;
- submit small, attributable batches only after a fresh live-table preflight.

This produced a larger and more reliable score jump than broad donor blending. The key is that the batch is taskwise, validated, and attributable, not a blind full-package replacement.


## Attribution for the May 31 improvements

Credit matters because the useful signals came from public work:

- `jsrdcht/glm-vs-opus-onnx-cost-opt-neurogolf-2026` published 10 optimized ONNX directly inside the notebook. All 10 transferred cleanly over my v69 baseline.
- `biohack44/neurogolf-2026-fp16-surgery-prune-blend-6115` exposed a stronger `task258` than the Massimiliano Part 2 output.
- `biohack44/neurogolf-new-best-public-v2-6080-ish` contributed fresh-hash `task076` and `task360` tails.
- Local cleanup after those grafts found small additional wins by deduplicating initializers and replacing selected mask multiplications with `Where`.

I did not use Rauff AI gameplay's large-looking local positives because those files hash-matched the old `afr1ste_6335_19` / 6323-style artifact family that already had online-negative evidence.


## The validation gate

Every promoted task in this artifact passed the same gate:

1. The ONNX file must load under the local current-rules evaluator.
2. Visible examples and generated checks must remain exact.
3. Cost must decrease enough to improve task points versus the current protected baseline.
4. The candidate must not be a known poisoned artifact hash unless there is fresh online evidence.
5. The Kaggle live table must show no pending row before submit.

This is why the final package uses only a small number of public-output files even though many public zips contain hundreds of different ONNX graphs.


## Why the rejected Rauff route matters

`rauffauzanrambe/ai-gameplay-neurogolf-predicted-submission` looked extremely attractive locally: tasks `066,219,255,285,319,324,366` summed to more than `+19` local points versus v71.

I still rejected it. Those exact hashes matched old high-score artifact files that had already failed online isolation. In NeuroGolf, local cost can be a false friend when the file comes from an old artifact family with mismatched hidden behavior or stale assumptions.

This is the main safety rule I recommend: **a positive local delta is necessary, but not sufficient, when the hash lineage is known bad.**


## Negative results that mattered

Several things were deliberately not submitted:

- Deep262003's first 250 downloaded output tasks had zero positives versus v70; a June 1 full-zip rerun versus v73 also had zero positives.
- Rauff `submission-extract`, `champions-arc-best`, and `best-score-kagglehub` outputs had zero positives versus v71.
- Rauff AI gameplay had large local positives but matched the old poisoned artifact family.
- June 1 checks of the Massimiliano Part 2 rerun, Biohack44 6115, Biohack44 v2, and JSRDCHT leftovers had no remaining positives once v73 already included the useful tasks.
- Post-v72 CSE, dead prune, no-op cast, no-op shape, neutral-op, and where-bool scans produced no useful candidates.
- Small-constant folding changed task219 but was not positive; post-v73 tiny tails on task076/task096 were kept for future attachment rather than submitted alone.

These misses are useful because they prevent wasting daily submission slots on repeated public-blend traps.


In [ ]:
print('Manifest task count:', len(manifest))
print('Total zip bytes:', out_path.stat().st_size)
print('Smallest ONNX files:')
manifest.sort_values('bytes').head(10)


## How I would continue from here

The next useful direction is not to blindly add more public outputs. I would continue this loop:

1. Refresh recent public notebooks and outputs.
2. Evaluate only hash-different files against the current protected baseline.
3. Reject known poisoned hashes even when local delta is large.
4. Attach clean micro-tails to larger validated candidates instead of burning slots one by one.
5. For structural work, focus on the remaining high-cost tasks and explain the graph pattern before submitting.

This route is slower than copying full zips, but it preserves attribution, avoids known traps, and keeps each score movement explainable.


## Closing note

This notebook is both an artifact and a playbook. It publishes the exact `6151.18` zip while keeping the attribution chain visible.

If you build on it, please keep the same standard: cite the public source, validate under the current rules, and record the negative hash families. The community gets more value from small exact wins and failed-cost notes than from another opaque zip.
